# Episode 01: Low-Rank Adaptation (LoRA) From First Principles

> **Getting Comfortable Series — Track 2: LLM Fine-Tuning & Alignment**  
> *"Don't simplify the topic. Simplify the path to understanding it."*

$$\text{Understand} \longrightarrow \text{Build} \longrightarrow \text{Break} \longrightarrow \text{Introduce Tools} \longrightarrow \text{Apply} \longrightarrow \text{Revisit}$$

---

### What You Will Learn Today
Fine-tuning an LLM does not require a million-dollar GPU cluster or memorizing complex configuration dictionaries. At its core, modern parameter-efficient fine-tuning (PEFT) is rooted in an elegant linear algebra insight: **Low-Rank Matrix Factorization**.

By the end of this notebook, you will:
1. **Understand linear transformations:** What actually happens when tokens multiply against weights ($y = x W^T$).
2. **Understand Matrix Rank from absolute scratch:** What "rank" really means using small $3 \times 3$ toy matrices you can compute in your head.
3. **The Low-Rank Factorization Trick:** How a $3 \times 3$ matrix of 9 numbers can be represented by 6 numbers, and how that scales to save millions of parameters in an LLM.
4. **The Two LoRA Secrets:** Why adapter $B$ is initialized to zeros and why we scale by $\frac{\alpha}{r}$.
5. **Build `LoRALinear` in Pure PyTorch:** Construct a working LoRA layer from scratch with zero third-party dependencies and verify that the base weights stay 100% frozen while only the adapter trains.
6. **Bridge to Production Tools:** Compare our hand-built PyTorch layer against Hugging Face `peft` and verify exact architectural parity.


## 0. Environment Setup & Sanity Check

Let's import PyTorch and NumPy. If you are on an Apple Silicon Mac, PyTorch will automatically leverage `mps` (Metal Performance Shaders). If you are on a standard machine or CPU, every cell in this notebook will still run instantaneously.


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Deterministic seed for reproducible intuition
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Active PyTorch Device: {device}")


---

## 1. Absolute Basics: What is a Weight Matrix in a Neural Network?

Before understanding how to *adapt* a weight matrix, let's refresh what a linear layer actually does to data.

In any transformer (whether GPT-4, Claude, or Qwen), every self-attention layer and MLP layer contains linear weight matrices ($W_{q}, W_{k}, W_{v}, W_{o}, W_{\text{gate}}, W_{\text{up}}, W_{\text{down}}$).

When an input vector $x$ (representing an embedded token) enters a linear layer:

$$y = x \cdot W^T$$

Let's look at this with tiny 3-dimensional numbers.
- Let input $x = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$ (a token with 3 embedding features).
- Let weight matrix $W$ be a $3 \times 3$ matrix representing what the neural network has learned during pre-training.


In [ ]:
# Input token representation (1 token, 3 features)
x = torch.tensor([[1.0, 2.0, 3.0]])

# Pre-trained weight matrix W_0 (3 output features, 3 input features)
W_0 = torch.tensor([
    [1.0, 0.0, 2.0],
    [0.0, 3.0, 1.0],
    [2.0, 1.0, 0.0]
])

# Linear transformation: y = x @ W_0.T
y = x @ W_0.T

print("Input x: ", x.numpy())
print("Weight W_0:\n", W_0.numpy())
print("Output y (x @ W_0.T):", y.numpy())


---

## 2. What on Earth is Matrix "Rank"? (The Intuition of Information)

The word "rank" sounds intimidating, but its meaning is simple:

> **The Rank of a matrix is the number of linearly independent rows or columns it contains.**  
> In plain English: **It is the number of truly unique, non-redundant directions of information inside the matrix.**

Let's compare two different $3 \times 3$ matrices to see this visually.

### Example A: A Full-Rank Matrix (Rank = 3)
In a full-rank matrix, every row points in a completely independent direction. No row can be formed by multiplying or adding other rows:

$$M_{\text{full}} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$
Here, all 3 dimensions are needed. Rank = 3.

---

### Example B: A Low-Rank Matrix (Rank = 1)
Now look at this matrix carefully:

$$M_{\text{low}} = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

Notice the pattern:
- Row 2 is simply **$2 \times \text{Row 1}$**.
- Row 3 is simply **$3 \times \text{Row 1}$**.

Even though there are **9 numbers** written in this matrix, there is really only **ONE unique pattern** (`[1, 2, 3]`), scaled by 1, 2, and 3.  
Because there is only 1 independent direction, its **Rank is 1**!


In [ ]:
# Let's verify this using PyTorch's matrix_rank function
M_full = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

M_low = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [3.0, 6.0, 9.0]
])

print(f"M_full rank: {torch.linalg.matrix_rank(M_full).item()} (Full rank: contains 3 independent dimensions)")
print(f"M_low rank:  {torch.linalg.matrix_rank(M_low).item()} (Low rank: contains only 1 independent direction!)")


---

## 3. The Low-Rank Factorization Trick (Decomposing $9$ numbers into $6$)

Because $M_{\text{low}}$ has Rank 1, **we don't need to store all 9 numbers!**

Any rank-1 matrix of size $3 \times 3$ can be factorized into the multiplication of two thin vectors:
- A column vector $B$ of size $3 \times 1$
- A row vector $A$ of size $1 \times 3$

Let's test this arithmetic by hand:

$$B = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}, \quad A = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$$

Multiply $B \times A$:

$$B \times A = \begin{bmatrix} 1 \cdot 1 & 1 \cdot 2 & 1 \cdot 3 \\ 2 \cdot 1 & 2 \cdot 2 & 2 \cdot 3 \\ 3 \cdot 1 & 3 \cdot 2 & 3 \cdot 3 \end{bmatrix} = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

**The Revelation:**
- To store the full matrix $M_{\text{low}}$, you needed $3 \times 3 = 9$ numbers.
- To store $B$ and $A$, you only need $3 + 3 = \mathbf{6}$ numbers.
- We achieved the exact same matrix while saving **$33\%$ of storage space**.


In [ ]:
B = torch.tensor([[1.0], [2.0], [3.0]])  # Shape (3, 1)
A = torch.tensor([[1.0, 2.0, 3.0]])       # Shape (1, 3)

# Matrix multiplication: (3, 1) @ (1, 3) -> (3, 3)
reconstructed = B @ A

print("B (3x1):\n", B.numpy())
print("A (1x3):\n", A.numpy())
print("\nB @ A (3x3):\n", reconstructed.numpy())
print("\nIs reconstructed identical to M_low?", torch.equal(reconstructed, M_low))


---

## 4. Why Does This Save LLM Fine-Tuning? (Scaling to 8 Billion Parameters)

In pre-trained foundation models, weight matrices are massive:
- In `Llama-3-8B` or `Qwen2.5-7B`, a single attention projection matrix has dimensions:
  $$d_{\text{in}} = 4096, \quad d_{\text{out}} = 4096$$

If you do **full fine-tuning**, you must compute gradients and Adam optimizer states for the full matrix update:
$$\Delta W \in \mathbb{R}^{4096 \times 4096} = 16,777,216 \text{ parameters } (\approx 67\text{ MB per matrix})$$

### The Edward Hu et al. (2021) Hypothesis:
> **When adapting a pre-trained model to a specific task or format, the weight change $\Delta W$ has an *extremely low intrinsic rank*.**  
> We do not need all 16.7 million degrees of freedom to teach a model a new tone, JSON format, or classification scheme!

If we set rank $r = 8$:
- Matrix $A \in \mathbb{R}^{8 \times 4096} = 32,768$ parameters
- Matrix $B \in \mathbb{R}^{4096 \times 8} = 32,768$ parameters
- **Total parameters to train:** $32,768 + 32,768 = \mathbf{65,536}$ parameters!

$$\frac{65,536}{16,777,216} = 0.0039 \implies \mathbf{99.61\% \text{ reduction in trainable parameters!}}$$


In [ ]:
def print_param_comparison(d_in, d_out, rank):
    full_params = d_in * d_out
    lora_params = (d_in * rank) + (rank * d_out)
    reduction = (1 - lora_params / full_params) * 100
    
    print(f"Layer Dimensions: {d_in} x {d_out}")
    print(f"  - Full Fine-Tuning Params: {full_params:,}")
    print(f"  - LoRA (rank={rank}) Params:       {lora_params:,}")
    print(f"  - Parameter Reduction:       {reduction:.2f}%")

print_param_comparison(d_in=4096, d_out=4096, rank=8)


---

## 5. The Two Critical Secrets of LoRA

In the paper *LoRA: Low-Rank Adaptation of Large Language Models*, the adapted layer is written as:

$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (B \cdot A) x$$

Notice two crucial details that every AI engineer must know:

### Secret #1: The Asymmetric Initialization
- **Matrix $A$** is initialized with random Gaussian noise: $A \sim \mathcal{N}\left(0, \frac{1}{r}\right)$.
- **Matrix $B$** is initialized to **EXACT ZERO**: $B = 0$.

> **Why initialize $B$ to zero?**  
> Because at step 0 (before training starts):
> $$\Delta W = B \cdot A = 0 \cdot A = 0$$
> This guarantees that at the very start of training, **the model behaves 100% identically to the base pre-trained model**. There is zero sudden jump or corrupted outputs before the first gradient step!

---

### Secret #2: The Scaling Factor $\frac{\alpha}{r}$
- Why multiply by $\alpha / r$ instead of just $B \cdot A$?
- $\alpha$ (alpha) is a constant (e.g. 16 or 32).
- When you experiment with different ranks (e.g. testing $r=4$ vs $r=16$), changing $r$ naturally changes the scale of the gradients. Multiplying by $\frac{\alpha}{r}$ stabilizes the magnitude of $\Delta W$, meaning **you don't have to re-tune your learning rate every time you change the rank**!


In [ ]:
# Visualizing Secret #1: Zero Initial Drift
r = 4
d_in, d_out = 3, 3

A_init = torch.randn(r, d_in) * 0.1
B_init = torch.zeros(d_out, r)

delta_W_step0 = B_init @ A_init
print("Delta W at Step 0 (B @ A):\n", delta_W_step0.numpy())
print("\nSum of all Delta W values:", delta_W_step0.sum().item(), "(Exactly zero drift!)")


---

## 6. Build: Implementing `LoRALinear` From Scratch in Pure PyTorch

Now let's build the complete, production-accurate `LoRALinear` layer from scratch.

Notice the mechanics:
1. It wraps a standard `nn.Linear` base layer.
2. It sets `base_layer.weight.requires_grad = False` (freezes the original weights).
3. It creates trainable parameters $A$ and $B$.
4. In `forward()`, it computes both paths and adds them together.


In [ ]:
class ToyLoRALinear(nn.Module):
    """
    A pure PyTorch implementation of Low-Rank Adaptation (LoRA).
    No third-party libraries required.
    """
    def __init__(self, base_layer: nn.Linear, r: int = 4, alpha: float = 8.0):
        super().__init__()
        self.base_layer = base_layer
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # 1. Freeze the original base layer weights!
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False

        # 2. Define LoRA low-rank matrices
        # A: (r, in_features) initialized with small Gaussian values
        self.lora_A = nn.Parameter(torch.empty(r, self.in_features))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # B: (out_features, r) initialized to EXACT ZERO
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Standard frozen base forward: x @ W0.T + bias
        base_out = self.base_layer(x)

        # LoRA forward: x @ (alpha/r * (B @ A).T)
        # We compute x @ A.T first to keep matrix operations thin!
        # (batch, seq, in) @ (in, r) -> (batch, seq, r)
        # (batch, seq, r) @ (r, out) -> (batch, seq, out)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

        return base_out + lora_out


---

## 7. Verify: Testing Gradient Isolation

Let's test our `ToyLoRALinear` layer:
1. Pass random inputs through it.
2. Compute a loss.
3. Call `loss.backward()`.
4. Inspect which parameters receive gradients and which stay completely frozen.


In [ ]:
# Create standard linear layer (e.g., hidden size 8 -> 8)
base_linear = nn.Linear(in_features=8, out_features=8, bias=False)

# Wrap it with our LoRA layer (rank=2, alpha=4)
lora_layer = ToyLoRALinear(base_linear, r=2, alpha=4.0)

# Create a sample batch of tokens: (batch_size=2, seq_len=3, hidden_dim=8)
sample_x = torch.randn(2, 3, 8)
sample_target = torch.randn(2, 3, 8)

# Forward pass
output = lora_layer(sample_x)
loss = F.mse_loss(output, sample_target)

# Backward pass
loss.backward()

print("=== Parameter Gradient Check ===")
print(f"Base Layer Weight requires_grad: {lora_layer.base_layer.weight.requires_grad}")
print(f"Base Layer Weight grad is None:   {lora_layer.base_layer.weight.grad is None}")
print(f"lora_A requires_grad:            {lora_layer.lora_A.requires_grad}")
print(f"lora_A grad computed?            {lora_layer.lora_A.grad is not None} (Shape: {lora_layer.lora_A.grad.shape})")
print(f"lora_B requires_grad:            {lora_layer.lora_B.requires_grad}")
print(f"lora_B grad computed?            {lora_layer.lora_B.grad is not None} (Shape: {lora_layer.lora_B.grad.shape})")


---

## 8. Build: A Complete 20-Step Training Loop

Let's watch our custom LoRA layer actually learn!
We will create a synthetic task where the base model is frozen, and only the LoRA adapter learns to map an input pattern to a target output.


In [ ]:
# Optimizer: We only pass lora_A and lora_B to AdamW!
optimizer = torch.optim.AdamW([lora_layer.lora_A, lora_layer.lora_B], lr=0.05)

print("Step | Loss")
print("---- | ---------")
for step in range(1, 21):
    optimizer.zero_grad()
    pred = lora_layer(sample_x)
    step_loss = F.mse_loss(pred, sample_target)
    step_loss.backward()
    optimizer.step()

    if step % 5 == 0 or step == 1:
        print(f"{step:4d} | {step_loss.item():.6f}")

print("\nTraining complete! The adapter learned the target mapping while the base weights remained 100% frozen.")


---

## 9. Merging the Adapter (Zero Latency Overhead in Production)

One common question engineers ask:
> *"Doesn't running two separate matrix paths ($W_0 x + B A x$) slow down inference in production?"*

The answer is **No!** Because matrix multiplication is distributive:

$$W_{\text{final}} = W_0 + \frac{\alpha}{r} (B \cdot A)$$

Before deploying to production, we can permanently **add the adapter directly into the base weights**!  
Once added, the adapter disappears into a single regular linear layer. Inference latency is **100% identical to the original un-tuned model**.


In [ ]:
with torch.no_grad():
    # Compute the final delta matrix
    delta_W = (lora_layer.lora_B @ lora_layer.lora_A) * lora_layer.scaling
    
    # Fuse it directly into the base weight!
    merged_weight = lora_layer.base_layer.weight + delta_W

print(f"Original Base Weight shape: {lora_layer.base_layer.weight.shape}")
print(f"Delta W shape:              {delta_W.shape}")
print(f"Merged Weight shape:         {merged_weight.shape}")
print("\nDeployment ready: the merged weight can now be loaded into any standard inference engine!")


---

## 10. Introduce Tools: Verifying Parity with Hugging Face `peft`

Now that you have built LoRA from mathematical first principles, let's look at how the industry-standard library Hugging Face `peft` implements the exact same logic.

When you install `peft` in production:
```bash
pip install peft transformers
```

You configure:
- `r`: The rank (identical to our `r=2`).
- `lora_alpha`: The scaling factor (identical to our `alpha=4.0`).
- `target_modules`: Which linear layers to wrap.


In [ ]:
try:
    from peft import LoraConfig, get_peft_model
    import transformers

    print("Hugging Face PEFT is installed!")
    # Define a tiny linear model
    toy_model = nn.Sequential(nn.Linear(8, 8, bias=False))
    
    # Configure PEFT LoraConfig
    config = LoraConfig(
        r=2,
        lora_alpha=4,
        target_modules=["0"], # wrap layer 0
        bias="none"
    )
    
    peft_toy = get_peft_model(toy_model, config)
    peft_toy.print_trainable_parameters()
    print("\nNotice the trainable parameter count matches our hand-built calculation exactly!")

except ImportError:
    print("Hugging Face 'peft' not installed in this environment.")
    print("To install for production: pip install peft transformers")
    print("Our pure PyTorch implementation above requires zero external packages and implements the identical mathematics!")


---

## 🎯 Summary: Your Mental Model for LoRA

| Concept | What It Actually Is | Why It Matters |
|---|---|---|
| **Weight Matrix $W_0$** | The pre-trained knowledge base ($d_{\text{out}} \times d_{\text{in}}$). | Kept completely **frozen**; zero gradient or optimizer memory overhead. |
| **Matrix Rank ($r$)** | The number of independent information directions. | Tasks like formatting or style only require $r=8$ or $r=16$ (low intrinsic rank). |
| **Adapter $A$** | Thin matrix $(r \times d_{\text{in}})$ initialized with Gaussian noise. | Projects high-dimensional inputs down to a low-dimensional bottleneck. |
| **Adapter $B$** | Thin matrix $(d_{\text{out}} \times r)$ initialized to **exact zeros**. | Ensures $\Delta W = 0$ at step 0; guarantees zero initial drift from pre-training. |
| **$\alpha / r$** | Scaling constant. | Keeps update scale stable across different rank experiments without retuning learning rates. |
| **Merging** | $W_{\text{final}} = W_0 + \frac{\alpha}{r}(BA)$. | Fuses updates into a single matrix for zero added inference latency in production. |

---

### What's Next?
In **Episode 02**, we tackle the #1 engineering bug in LLM fine-tuning: **Data formatting, Jinja chat templates, and completion-only loss masking**. You will see how unmasked prompt tokens cause models to regurgitate instructions, and how to fix it using `-100` label masking.

👉 Open [`comfort-refresh.md`](./comfort-refresh.md) anytime for a 3-minute recap!
